# Get the brain data

In [1]:
%load_ext autoreload
%autoreload 2

In [6]:
import os

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.stats as st
import scvi
import seaborn as sns
import xarray as xr
from anndata import AnnData
from rich import print

import nichevi

scvi.settings.seed = 34

Global seed set to 34


In [7]:
N_LAYERS = 1
N_LATENT = 10
LIKELIHOOD = "nb"
K_NN = 5
N_HEADS = 1
N_EPOCHS_NICHEVI = 1

from scvi.data import synthetic_iid

from nichevi import nicheSCVI

adata = synthetic_iid(
    batch_size=256,
    n_genes=100,
    n_proteins=0,
    n_regions=0,
    n_batches=2,
    n_labels=3,
    dropout_ratio=0.5,
    generate_coordinates=True,
    sparse_format=None,
    return_mudata=False,
)

adata.obsm["qz1_m"] = np.random.normal(size=(adata.shape[0], N_LATENT))
adata.layers["counts"] = adata.X.copy()

setup_kwargs = {
    "labels_key": "labels",
    "cell_coordinates_key": "coordinates",
    "expression_embedding_key": "qz1_m",
    "expression_embedding_niche_key": "qz1_m_niche_ct",
    "niche_composition_key": "neighborhood_composition",
    "niche_indexes_key": "niche_indexes",
    "niche_distances_key": "niche_distances",
}

nicheSCVI.preprocessing_anndata(
    adata,
    k_nn=K_NN,
    sample_key="batch",
    **setup_kwargs,
)

nicheSCVI.setup_anndata(
    adata,
    layer="counts",
    batch_key="batch",
    **setup_kwargs,
)

niche_setup = {
    "r1_kl1_c1_n1": {
        "cell_rec_weight": 1.0,
        "niche_rec_weight": 1.0,
        "niche_compo_weight": 1.0,
        "latent_kl_weight": 1.0,
    },
    "r0_kl0_c0_n0": {
        "cell_rec_weight": 0.0,
        "niche_rec_weight": 0.0,
        "niche_compo_weight": 0.0,
        "latent_kl_weight": 0.0,
    },
}

setup_dict = niche_setup["r0_kl0_c0_n0"]

nichevae = nicheSCVI(
    adata,
    cell_rec_weight=setup_dict["cell_rec_weight"],
    latent_kl_weight=setup_dict["latent_kl_weight"],
    niche_rec_weight=setup_dict["niche_rec_weight"],
    compo_rec_weight=setup_dict["niche_compo_weight"],
    gene_likelihood=LIKELIHOOD,
    n_layers=N_LAYERS,
    n_heads=N_HEADS,
    n_layers_niche=1,
    n_layers_compo=1,
    n_hidden_niche=48,
    n_hidden_compo=48,
    n_latent=N_LATENT,
    use_batch_norm="both",
    use_layer_norm="none",
)

Saved niche_indexes and niche_distances in adata.obsm

Saved niche_composition in adata.obsm

Saved qz1_m_niche_ct in adata.obsm

2024-04-16 19:04:46.124335: W external/xla/xla/service/platform_util.cc:198] unable to create StreamExecutor for CUDA:0: failed initializing StreamExecutor for CUDA device ordinal 0: INTERNAL: failed call to cuDevicePrimaryCtxRetain: CUDA_ERROR_DEVICE_UNAVAILABLE: CUDA-capable device(s) is/are busy or unavailable


INFO     Using column names from columns of adata.obsm['neighborhood_composition']                                 
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        
INFO     Generating sequential column names                                                                        


In [9]:
from scvi import REGISTRY_KEYS

from nichevi import NICHEVI_REGISTRY_KEYS


In [10]:
label_key = nichevae.adata_manager.get_from_registry(REGISTRY_KEYS.LABELS_KEY)
sample_key = nichevae.adata_manager.get_from_registry(NICHEVI_REGISTRY_KEYS.SAMPLE_KEY)
cell_coordinates_key = nichevae.adata_manager.get_from_registry(
    NICHEVI_REGISTRY_KEYS.CELL_COORDINATES_KEY
)

In [19]:
label_key

np.unique(label_key)

array([0, 1, 2], dtype=int8)

In [24]:
(label_key == 0).shape

(512, 1)

In [27]:
mask = np.squeeze(label_key == 0, axis=1)
cell_coordinates_key[mask].shape

(175, 2)

In [17]:
adata.obs["labels"]

0      label_0
1      label_1
2      label_1
3      label_2
4      label_1
        ...   
507    label_2
508    label_1
509    label_0
510    label_0
511    label_2
Name: labels, Length: 512, dtype: category
Categories (3, object): ['label_0', 'label_1', 'label_2']

In [4]:
from dataclasses import dataclass
from typing import Optional


@dataclass(frozen=True)
class PARAMS:
    """set all params for our analysis"""

    DATA_FOLDER: str = (
        "/home/labs/nyosef/Collaboration/SpatialDatasets/xenium/human_breast_cancer/"
    )
    # DATA_FOLDER: str = "/home/nathanlevy/sda/Data/"
    # DATA_FOLDER: str = "/home/labs/nyosef/Collaboration/SpatialDatasets/merfish/whole_brain_atlas_zhuang/"


    DATA_FILE: str = "xenium_breast_cancer_S1_R1_2.h5ad"
    # DATA_FILE: str = "adata_M1_M2_core_6_sections.h5ad"

    # # for the data
    # COORDS: str = "spatial"
    # BATCH: str = "brain_section_label"
    # SAMPLE: str = "brain_section_label"
    # CELL_TYPE: str = "cell_type"
    # NICHE: str = "major_brain_region"

    # for the data
    COORDS: str = "spatial"
    BATCH: str = "sample"
    SAMPLE: str = "sample"
    CELL_TYPE: str = "cell_type"
    NICHE: str = "kmeans_alpha_5"

    # for scVI
    N_LAYERS: int = 1
    N_LATENT: int = 10
    LIKELIHOOD: str = "nb"
    ######################
    N_EPOCHS_SCVI: int = 400
    BATCH_SIZE: int = 128
    OPTIMIZER: str = "Adam"
    ######################
    WEIGHT_DECAY: float = 1e-6
    KL_WARMUP: Optional[int] = None

    # for nicheVI
    K_NN: int = 20  # TODO try also less cells per niche, like 12
    N_LAYERS_NICHE: int = 1
    N_LAYERS_COMPO: int = 1
    N_HIDDEN: int = 128
    N_HIDDEN_COMPO: int = 128  # TODO make it adaptable
    N_HIDDEN_NICHE: int = 128
    N_LATENT_NICHEVI: int = 10
    ATTENTION_DECODER: bool = True
    N_HEADS: int = 2
    ######################
    N_EPOCHS_NICHEVI: int = 5  # FAKE
    REDUCE_LR_ON_PLATEAU: bool = True
    LR: float = 1e-4
    SPATIAL_WARMUP: Optional[int] = 10
    MIN_SPATIAL_WEIGHT: float = 1.0
    MAX_SPATIAL_WEIGHT: float = 10

    # +batch norm/layer norm

    CELL_TYPES_FOR_NICHES = None

    CELL_TYPES_FOR_STRATIFICATION = [
        # "DCIS_2",
        "Macrophages_1",
        # "Unlabeled",
        # "Invasive_Tumor",
        # "Stromal",
        # "DCIS_1",
        # "Myoepi_ACTA2+",
        "CD8+_T_Cells",
        # "Endothelial",
        # "Prolif_Invasive_Tumor",
        # "T_Cell_&_Tumor_Hybrid",
        # "Mast_Cells",
        "CD4+_T_Cells",
        "B_Cells",
        "Macrophages_2",
        # "Stromal_&_T_Cell_Hybrid",
        # "Perivascular-Like",
        "LAMP3+_DCs",  # PUT THE DCS TOGETHER
        "IRF7+_DCs",
        # "Myoepi_KRT15+",
    ]


setup = PARAMS()

In [ ]:
data_dir, data_file = setup.DATA_FOLDER, setup.DATA_FILE

data_file_name = os.path.splitext(data_file)[0]
print(data_file_name)

path_to_save = os.path.join("../niche-VI-experiments/checkpoints", data_file_name)
os.makedirs(path_to_save, exist_ok=True)

adata = ad.read_h5ad(os.path.join(data_dir, data_file))
print(adata)

In [ ]:
# adata.var_names = adata.var["gene_name"]

# Run scvi-nichevi

In [ ]:
scvi_is_trained = True
nichevi_is_trained = False
save_scvi = False
save_nichevi = True

print("scVI trained: ", scvi_is_trained, "nicheVI trained: ", nichevi_is_trained)


history_setup = {}

if scvi_is_trained is False:
    scvi.model.SCVI.setup_anndata(
        adata,
        layer="counts",
        batch_key=setup.BATCH,
    )

    scvivae = scvi.model.SCVI(
        adata,
        gene_likelihood=setup.LIKELIHOOD,
        n_layers=setup.N_LAYERS,
        n_latent=setup.N_LATENT,
    )

    scvivae.train(
        max_epochs=setup.N_EPOCHS_SCVI,
        train_size=0.8,
        validation_size=0.2,
        batch_size=setup.BATCH_SIZE,
        plan_kwargs={
            "n_epochs_kl_warmup": setup.KL_WARMUP,
            "weight_decay": setup.WEIGHT_DECAY,
            "optimizer": setup.OPTIMIZER,
        },
        # trainer_kwargs=dict(check_val_every_n_epoch=1),
        early_stopping=True,
    )

    if save_scvi:
        scvivae.save(
            dir_path=path_to_save
            + "/scvivae_E"
            + str(setup.N_EPOCHS_SCVI)
            # + "_"
            # + str(setup.N_LAYERS)
            # + "L"
            + ".pt",
            save_anndata=False,
        )

if scvi_is_trained:
    scvivae = scvi.model.SCVI.load(
        dir_path=path_to_save + "/scvivae_E" + str(setup.N_EPOCHS_SCVI) + ".pt",
        adata=adata,
    )

adata.obsm["X_scVI"] = scvivae.get_latent_representation(batch_size=setup.BATCH_SIZE)
# SCVI_MDE_KEY = "X_scVI_MDE"
# adata.obsm[SCVI_MDE_KEY] = scvi.model.utils.mde(adata.obsm["X_scVI"])

# adata.write_h5ad(
#     os.path.join(path_to_save, data_file_name + "_scVI.h5ad"),
# )

history_setup["X_scVI"] = scvivae.history


print("scVI done...")

: 

In [ ]:
df_DE_scvi = scvivae.differential_expression(
    groupby=setup.CELL_TYPE,
    group1="microglial cell",
    # group2="CD4+_T_Cells",
    mode="change",
    batch_correction=False,
)

: 

In [ ]:
# df_DE_nichevi.head(30)

: 

https://pytorch-lightning.readthedocs.io/en/1.5.10/advanced/profiler.html#built-in-checks

In [ ]:
from lightning.pytorch.callbacks import DeviceStatsMonitor

device_stats = DeviceStatsMonitor()
# trainer = Trainer(callbacks=[device_stats])

: 

In [ ]:
niche_setup = {
    "scvi_r1_kl1_s10": {
        "cell_rec_weight": 1,
        "latent_kl_weight": 1.0,
        "spatial_weight": 1,
        "n_latent": setup.N_LATENT_NICHEVI,
        "n_layers_niche": setup.N_LAYERS_NICHE,
        "n_layers_compo": setup.N_LAYERS_COMPO,
        "n_hidden_niche": setup.N_HIDDEN_NICHE,
        "n_hidden_compo": setup.N_HIDDEN_COMPO,
        "niche_expression": "pca",
        "optimizer": setup.OPTIMIZER,
    },
}

: 

In [ ]:
for setting in niche_setup.keys():
    print("[bold green]" + setting + "[/bold green]")
    setup_dict = niche_setup[setting]

    # preprocessing function to populate adata.obsm with the keys 'neighborhood_composition',
    # 'qz1_m', 'qz1_var', 'niche_indexes', 'niche_distances', 'qz1_m_niche_knn', 'qz1_var_niche_knn', 'qz1_m_niche_ct',
    # 'qz1_var_niche_ct'

    path_to_save_nichevae = path_to_save + "/nichevae_" + setting + "_" + str(setup.N_EPOCHS_NICHEVI) + ".pt"


    if setup_dict["niche_expression"] == "pca":
        NICHE_LIKELIHOOD = "gaussian"
        adata.obsm["qz1_m"] = adata.obsm["X_pca"]

    if setup_dict["niche_expression"] == "scvi":
        NICHE_LIKELIHOOD = "gaussian"
        adata.obsm["qz1_m"] = adata.obsm["X_scVI"]

    if setup_dict["niche_expression"] == "gaussian_counts":
        NICHE_LIKELIHOOD = "gaussian"
        adata.obsm["qz1_m"] = adata.X.toarray()

    nichevi.nicheSCVI.preprocessing_anndata(
        adata,
        niche_composition_key="neighborhood_composition",
        niche_indexes_key="niche_indexes",
        niche_distances_key="niche_distances",
        niche_type_key="niche_type",
        niche_treshold=None,
        cell_type_for_niches=None,
        label_key=setup.CELL_TYPE,
        sample_key=setup.SAMPLE,
        cell_coordinates_key=setup.COORDS,
        k_nn=setup.K_NN,
        latent_mean_key="qz1_m",
        latent_mean_niche_key="qz1_m_niche_ct",
    )

    if setup_dict["niche_expression"] == "poisson_counts":
        adata.obsm["qz1_m_niche_ct"] = np.ceil(adata.obsm["qz1_m_niche_ct"])

    nichevi.nicheSCVI.setup_anndata(
        adata,
        layer="counts",
        batch_key=setup.BATCH,
        labels_key=setup.CELL_TYPE,
        niche_composition_key="neighborhood_composition",
        niche_indexes_key="niche_indexes",
        niche_distances_key="niche_distances",
        latent_mean_key="qz1_m",
        latent_mean_ct_key="qz1_m_niche_ct",
    )

    if nichevi_is_trained is False:
        if save_nichevi:
            # check if path_to_save_nichevae exists
            if os.path.exists(path_to_save_nichevae):
                print("Model already exists...")
                # get out of the loop
                continue

        nichevae = nichevi.nicheSCVI(
            adata,
            cell_rec_weight=setup_dict["cell_rec_weight"],
            latent_kl_weight=setup_dict["latent_kl_weight"],
            spatial_weight=setup_dict["spatial_weight"],
            niche_likelihood=NICHE_LIKELIHOOD,
            gene_likelihood=setup.LIKELIHOOD,
            n_layers=setup.N_LAYERS,
            n_heads=setup.N_HEADS,
            n_layers_niche=setup_dict["n_layers_niche"],
            n_layers_compo=setup_dict["n_layers_compo"],
            n_hidden_niche=setup_dict["n_hidden_niche"],
            n_hidden_compo=setup_dict["n_hidden_compo"],
            n_latent=setup_dict["n_latent"],
            use_batch_norm="both",
            use_layer_norm="none",
        )

        nichevae.train(
            max_epochs=setup.N_EPOCHS_NICHEVI,
            train_size=0.8,
            validation_size=0.2,
            early_stopping=True,
            check_val_every_n_epoch=1,
            batch_size=setup.BATCH_SIZE,
            plan_kwargs={
                "lr": setup.LR,
                "n_epochs_kl_warmup": setup.KL_WARMUP,
                "n_epochs_spatial_warmup": setup.SPATIAL_WARMUP,
                # n_epochs_kl_warmup=N_EPOCHS_NICHEVI,
                "min_spatial_weight": setup.MIN_SPATIAL_WEIGHT,
                "max_spatial_weight": setup.MAX_SPATIAL_WEIGHT,
                "optimizer": setup.OPTIMIZER,
                "weight_decay": setup.WEIGHT_DECAY,
                "reduce_lr_on_plateau": setup.REDUCE_LR_ON_PLATEAU,
            },
            # callbacks=[device_stats],
            profiler="simple",
        )

        if save_nichevi:
            nichevae.save(
                dir_path=path_to_save_nichevae,
                save_anndata=False,
            )

    if nichevi_is_trained:
        nichevae = nichevi.nicheSCVI.load(
            dir_path=path_to_save_nichevae,
            adata=adata,
        )

    history_setup[setting] = nichevae.history
    adata.obsm[setting + "_X_nicheVI"] = nichevae.get_latent_representation(batch_size=1024)

    # NICHEVI_MDE_KEY = setting + "_X_nicheVI_MDE"
    # adata.obsm[NICHEVI_MDE_KEY] = scvi.model.utils.mde(
    #     adata.obsm[setting + "_X_nicheVI"]
    # )

: 

In [ ]:
df_DE_nichevi = nichevae.differential_expression(
    groupby=setup.CELL_TYPE,
    group1="microglial cell",
    group2="macrophage",
    mode="change",
    batch_correction=False,
    compute_neighborhood_de=True,
    sample_key=setup.SAMPLE,
    cell_coordinates_key=setup.COORDS,
    label_key=setup.CELL_TYPE,
    radius=50,
    k_nn=None,
)

: 

In [ ]:
df_DE_nichevi["group1_group2"].head()

: 

In [ ]:
df_DE_nichevi["group1_niche1"].head()

: 

In [ ]:
df_A = df_DE_nichevi["group1_group2"]
df_B = df_DE_nichevi["group1_niche1"]


# Create a dictionary to store the DataFrames
dfs = {"group1_group2": df_A, "group1_niche1": df_B}

# Create an xarray Dataset from the dictionary
ds = xr.Dataset(dfs)

# Make the 'condition' dimension a coordinate
# ds = ds.expand_dims("condition")
# ds = ds.swap_dims({"condition": "condition"})
ds.coords["condition"] = ["group1_group2", "group1_niche1"]
# ds = ds.rename({"condition_A": "group1_group2", "condition_B": "group1_niche1"})
ds = ds.rename({"dim_1": "columns"})

print(ds)

: 

In [ ]:
ds.sel(columns="bayes_factor", gene_name="Acta2").to_dataframe()

: 

In [ ]:
print(*nichevae.history.keys())

: 

In [ ]:
# adata.obsm["niche_attention"] = nichevae.get_niche_attention(batch_size=1024)

: 

In [ ]:
nichevae.summary_stats

: 

In [ ]:
adata.obsm["niche_distances"][:, :5]
# adata.obsm["niche_indexes"].shape

: 

In [ ]:
# # take 'adata.obsm[neighborhood_composition'] and group by adata.obs["kmeans_alpha_5"] discrete values, then plot the mean of each group:
# mean_prop_per_cluster = (
#     adata.obsm["neighborhood_composition"].groupby(adata.obs[setup.NICHE]).mean()
# )


# # okay so not a barplot but a bar that is the contribution of each cell type for each cluster

# # put the legend outside the plot

# mean_prop_per_cluster.plot.bar(stacked=True, figsize=(15, 8), cmap="tab20")

# plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))

: 

# COUNT CORRUPTION - and CONNECTIVITY updates

In [ ]:
from nichevi.differential_expression import adjusted_nearest_neighbors  # corrupt_counts

: 

In [ ]:
# corrupt_counts(
#     adata,
#     "niche_indexes",
#     "niche_distances",
#     use_layer="counts",
#     save_weights=True,
#     bandwidth="median",
#     k_nn=5,
# )

# adata.obsm["corruption_weights"]  # .sum(axis=1).max()

: 

In [ ]:
A = adjusted_nearest_neighbors(
    adata,
    setup.SAMPLE,
    setup.COORDS,
    setup.CELL_TYPE,
    radius=50,
    k_nn=None,
    return_sparse=True,
)

A

: 

In [ ]:
A.sum(axis=1).max()

: 

In [ ]:
test_idx = 17

: 

In [ ]:
cell_types = adata.obs[setup.CELL_TYPE].values

cell_types[test_idx]

: 

In [ ]:
np.sort(np.nonzero(A[test_idx])[0])

: 

In [ ]:
len(np.nonzero(A[test_idx])[0])

: 

In [ ]:
np.sort(adata.obsm["niche_indexes"][test_idx])

: 

In [ ]:
cell_types[np.sort(adata.obsm["niche_indexes"][test_idx])]

: 

In [ ]:
# Get the number of non-zero entries per row
row_counts = np.diff(A.indptr)

# Create the histogram
fig, ax = plt.subplots()
ax.hist(row_counts, bins=np.max(row_counts) + 1, density=True)
ax.set_xlabel("Number of non-zero entries")
ax.set_ylabel("Number of rows")
ax.set_title("Histogram of non-zero entries per row")
plt.show()

: 

In [ ]:
adata.obsm["neighborhood_composition"].iloc[250750]

: 

# GENERAL MICROGLIA DE

Microglia vs all 

Wilcoxon test

Then filter with min_in_group_fraction=0.2, max_out_group_fraction=0.2

Select LFC>2

Then ask Claude/GPT whih genes are actually microglia markers

In [ ]:
df_DE_scvi

: 

In [ ]:
df_DE_nichevi = nichevae.differential_expression(
    groupby=setup.CELL_TYPE,
    group1="microglial cell",
    group2="macrophage",
    mode="change",
    batch_correction=False,
    compute_neighborhood_de=True,
    sample_key=setup.SAMPLE,
    cell_coordinates_key=setup.COORDS,
    label_key=setup.CELL_TYPE,
    radius=50,
    k_nn=None,
)

: 

In [ ]:
df_DE_nichevi.keys()

: 

: 

In [ ]:
if "log1p" in adata.uns.keys():
    adata.uns.pop("log1p")

sc.tl.rank_genes_groups(
    adata,
    groupby=setup.CELL_TYPE,
    # method="wilcoxon",
    method="t-test",
    key_added="rank_genes_groups",
    pts=True,
    use_raw=False,
    groups=["microglial cell"],
)

sc.pl.rank_genes_groups_dotplot(
    adata,
    groupby=setup.CELL_TYPE,
    key="rank_genes_groups",
    # save="nicheVI_rank_genes_groups_dotplot.pdf",
    show=False,
    use_raw=False,
)

: 

In [ ]:
sc.get.rank_genes_groups_df(adata, key="rank_genes_groups", group="microglial cell", log2fc_min=2)  # ['names'].tolist()

: 

### CLAUDE

Here is my final list of potential microglia marker genes from the provided list, along with a brief description of each gene:

1. `Ctss` (Cathepsin S) - A lysosomal cysteine protease highly expressed in microglia and involved in antigen presentation and immune responses.

2. `Ptprc` (CD45) - A transmembrane protein tyrosine phosphatase expressed on microglia and other cells of hematopoietic origin, commonly used as a pan-leukocyte marker.

3. `Itga9` (Integrin alpha 9) - An integrin subunit involved in cell-cell and cell-matrix interactions, highly expressed in microglia and implicated in their migration and activation.

4. `Ctsc` (Cathepsin C) - Another lysosomal cysteine protease expressed in microglia and involved in immune functions.

5. `Sall1` (Spalt-Like Transcription Factor 1) - A transcription factor implicated in microglia development and function.

6. `Mafb` (Musculoaponeurotic Fibrosarcoma Oncogene Homolog B) - A transcription factor involved in the regulation of microglia-specific gene expression and function.

7. `Lcp1` (Lymphocyte Cytosolic Protein 1) - A gene involved in T-cell activation and reportedly highly expressed in microglia.

8. `Tgfbr2` (Transforming Growth Factor Beta Receptor 2) - A receptor for TGF-beta, which plays a role in microglia activation and neuroinflammation.

9. `Fli1` (Friend Leukemia Integration 1 Transcription Factor) - A transcription factor involved in the regulation of microglia gene expression and function.

10. `Hpgd` (Hydroxyprostaglandin Dehydrogenase 15-(NAD)) - An enzyme involved in prostaglandin metabolism, which is important for microglia inflammatory responses.

11. `Ets1` (ETS Proto-Oncogene 1, Transcription Factor) - A transcription factor implicated in microglia activation and inflammatory responses.

Please note that while these genes are widely associated with microglia, their expression levels and roles can vary depending on the specific context and experimental conditions. Additionally, some of the other genes in the provided list may also have relevance to microglia biology, although they may not be considered classic microglia markers.


### GPT

Certainly! Here's a bullet list of genes along with their descriptions:

- Ctss: Cathepsin S (Ctss) is a lysosomal cysteine protease involved in antigen presentation and immune responses.
- Sall1: Sall1 is a transcription factor involved in various developmental processes, including kidney and central nervous system development.
- Mafb: Mafb is a transcription factor known to regulate macrophage and microglial development and function.
- Maf: Maf is a transcription factor that plays a role in the regulation of immune responses and inflammation.
- Lcp1: Lcp1 (Lymphocyte cytosolic protein 1) is involved in the regulation of actin dynamics and cell migration, including microglial motility.
- Fyb: Fyb (FYN binding protein) is involved in T cell receptor signaling and could have roles in microglial activation and function.
- Ptprc: Ptprc (Protein tyrosine phosphatase, receptor type C), also known as CD45, is a transmembrane protein tyrosine phosphatase expressed on leukocytes, including microglia.
- Tmem176b: Tmem176b is a transmembrane protein associated with lysosomal function and has been implicated in microglial activation.
- Sall3: Sall3 is a transcription factor involved in neural development and has been reported to be expressed in microglia.
- Tmem176a: Tmem176a is a transmembrane protein expressed in microglia and implicated in lysosomal function and immune responses.

In [ ]:
microglia_markers_naive = sc.get.rank_genes_groups_df(
    adata, key="rank_genes_groups", group="microglial cell", log2fc_min=2
)["names"].tolist()

marker_indices_naive = [adata.var_names.get_loc(gene) for gene in microglia_markers_naive if gene in adata.var_names]

microglia_markers_claude = [
    "Ctss",  # Cathepsin S
    "Ptprc",  # CD45
    "Itga9",  # Integrin alpha 9
    "Ctsc",  # Cathepsin C
    "Sall1",  # Spalt-Like Transcription Factor 1
    "Mafb",  # Musculoaponeurotic Fibrosarcoma Oncogene Homolog B
    "Lcp1",  # Lymphocyte Cytosolic Protein 1
    "Tgfbr2",  # Transforming Growth Factor Beta Receptor 2
    "Fli1",  # Friend Leukemia Integration 1 Transcription Factor
    "Hpgd",  # Hydroxyprostaglandin Dehydrogenase 15-(NAD)
    "Ets1",  # ETS Proto-Oncogene 1, Transcription Factor
]

marker_indices_claude = [adata.var_names.get_loc(gene) for gene in microglia_markers_claude if gene in adata.var_names]

microglia_markers_chatgpt = [
    "Ctss",  # Cathepsin S (Ctss) is a lysosomal cysteine protease involved in antigen presentation and immune responses.
    "Sall1",  # Sall1 is a transcription factor involved in various developmental processes, including kidney and central nervous system development.
    "Mafb",  # Mafb is a transcription factor known to regulate macrophage and microglial development and function.
    "Maf",  # Maf is a transcription factor that plays a role in the regulation of immune responses and inflammation.
    "Lcp1",  # Lcp1 (Lymphocyte cytosolic protein 1) is involved in the regulation of actin dynamics and cell migration, including microglial motility.
    "Fyb",  # Fyb (FYN binding protein) is involved in T cell receptor signaling and could have roles in microglial activation and function.
    "Ptprc",  # Ptprc (Protein tyrosine phosphatase, receptor type C), also known as CD45, is a transmembrane protein tyrosine phosphatase expressed on leukocytes, including microglia.
    "Tmem176b",  # Tmem176b is a transmembrane protein associated with lysosomal function and has been implicated in microglial activation.
    "Sall3",  # Sall3 is a transcription factor involved in neural development and has been reported to be expressed in microglia.
    "Tmem176a",  # Tmem176a is a transmembrane protein expressed in microglia and implicated in lysosomal function and immune responses.
]

: 

In [ ]:
adata

: 

In [ ]:
import numpy as np

# Assuming adata is your AnnData object and marker_indices_naive are the indices of markers
# adata_markers = adata[:, marker_indices_naive].copy()
adata_markers = adata.copy()

# Get the indices for microglial and non-microglial cells
microglial_indices = adata_markers.obs[setup.CELL_TYPE] == "microglial cell"
non_microglial_indices = ~microglial_indices

# compute the average expression of the markers for cells in the microglial cell group
mean_dict = {
    "uncorrupted_microglia": np.array(adata_markers.layers["uncorrupted_counts"][microglial_indices].mean(axis=0))[0],
    "corrupted_microglia": np.array(adata_markers.layers["corrupted_counts"][microglial_indices].mean(axis=0))[0],
    "uncorrupted_ref:": np.array(adata_markers.layers["uncorrupted_counts"][non_microglial_indices].mean(axis=0))[0],
    "corrupted_ref": np.array(adata_markers.layers["corrupted_counts"][non_microglial_indices].mean(axis=0))[0],
}

# Create a DataFrame
df = pd.DataFrame(mean_dict, index=adata_markers.var_names)
df.T

: 

In [ ]:
ttest_result = st.ttest_rel(
    a=np.array(adata_markers.layers["uncorrupted_counts"][microglial_indices].toarray()),
    b=np.array(adata_markers.layers["corrupted_counts"][microglial_indices].toarray()),
    alternative="greater",
)

df["pval_mean-exp"] = ttest_result.pvalue

df["adj_pval_mean-exp"] = st.false_discovery_control(ttest_result.pvalue, method="bh")

: 

In [ ]:
df[df["uncorrupted_microglia"] > df["corrupted_microglia"]].sort_values(by="adj_pval_mean-exp")

: 

In [ ]:
df["pval_mean-exp"].hist()

: 

In [ ]:
["Ctss", "Sall1", "Mafb", "Maf", "Lcp1", "Fyb", "Ptprc", "Nfam1", "Blnk", "Upk1b"]

: 

In [ ]:
# sc.get.rank_genes_groups_df(adata, key="rank_genes_groups", group="microglial cell", log2fc_min=2) #['names'].tolist()

: 

# Compare with corrupted counts

# CLUSTER NICHEVI LATENT

In [ ]:
nichevi_latent_key = "scvi_r1_kl1_s10_X_nicheVI"
banksy_key = "banksy_pc_20_lambda_0.4"


cell_type_for_study = "microglial cell"
# cell_type_for_study = "macrophage"

leiden_resolution = 0.6

corrupt_counts(
    adata,
    "niche_indexes",
    "niche_distances",
    use_layer="counts",
    save_weights=True,
    bandwidth="median",
    k_nn=5,
    target_sum=1e4,
    spatial_weight=1.0,
    log1p=True,
)

# Select the cell type to analyze
adata_CD4 = adata[adata.obs[setup.CELL_TYPE].isin([cell_type_for_study])].copy()

# Compute the MDEs
adata_CD4.obsm["MDE_CD4_scVI"] = scvi.model.utils.mde(adata_CD4.obsm["X_scVI"])

adata_CD4.obsm["MDE_CD4_nicheVI"] = scvi.model.utils.mde(adata_CD4.obsm[nichevi_latent_key])

adata_CD4.obsm["MDE_CD4_simVI"] = scvi.model.utils.mde(adata_CD4.obsm["simvi_both_08"])

adata_CD4.obsm["MDE_CD4_banksy"] = scvi.model.utils.mde(adata_CD4.obsm[banksy_key])


# LEIDEN
sc.pp.neighbors(adata_CD4, use_rep="X_scVI")
sc.tl.leiden(adata_CD4, resolution=leiden_resolution, key_added="leiden_scvi")

sc.pp.neighbors(adata_CD4, use_rep="simvi_both_08")
sc.tl.leiden(adata_CD4, resolution=leiden_resolution, key_added="leiden_simvi")

sc.pp.neighbors(adata_CD4, use_rep=nichevi_latent_key)
sc.tl.leiden(adata_CD4, resolution=leiden_resolution, key_added="leiden_nichevi")

sc.pp.neighbors(adata_CD4, use_rep=banksy_key)
sc.tl.leiden(adata_CD4, resolution=leiden_resolution, key_added="leiden_banksy")

: 

In [ ]:
from nichevi.spatial_analysis import compute_jaccard

proportions = compute_jaccard(adata_CD4, "leiden_nichevi", setup.NICHE)

: 

In [ ]:
idx = pd.Index(
    [
        "Thalamus",
        "Hippocampus",
        "Isocortex",
        "Hypothalamus",
        "Striatum",
        "Pallidum",
        "Fiber_tracts",
        "Ventricular_systems",
        "Olfactory",
        "Cortical_subplate",
    ],
    dtype="object",
)

fs = 20
CLUSTERMAP = True
plt.figure(figsize=(10, 6))

if CLUSTERMAP:
    fig = sns.clustermap(
        data=proportions.loc[idx],
        cmap="Blues",
        row_cluster=False,
        col_cluster=True,
        annot=False,
        fmt=".2f",
        vmax=0.6,
        figsize=(10, 6),
    )
    # fig.fig.suptitle('Jaccard similarity')
    ax = fig.ax_heatmap

    ax.grid(False)
    # ax.set_xlabel('Samples')
    # ax.set_ylabel('Samples')

else:
    # Create a heatmap
    ax = sns.heatmap(
        proportions.loc[idx],
        annot=False,
        cmap="Blues",
        cbar=True,
        linewidths=0.5,
        fmt=".2%",
        annot_kws={"size": 18},
        vmin=0,  # Set minimum colorbar limit
        vmax=0.6,  # Set maximum colorbar limit
    )

# Adjust xticks and yticks fontsize
ax.set_xticklabels(ax.get_xticklabels(), fontsize=fs)
ax.set_yticklabels(ax.get_yticklabels(), fontsize=fs)

# Adjust colorbar fontsize
cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=fs)

plt.grid(False)
# plt.savefig(f"{figure_dir}clustermap_scvi.png", bbox_inches="tight", dpi=1000)
plt.show()

: 

In [ ]:
# take 'adata.obsm[neighborhood_composition'] and group by adata.obs["kmeans_alpha_5"] discrete values, then plot the mean of each group:
mean_prop_per_cluster = adata_CD4.obsm["neighborhood_composition"].groupby(adata_CD4.obs["leiden_nichevi"]).mean()


# okay so not a barplot but a bar that is the contribution of each cell type for each cluster

# put the legend outside the plot

mean_prop_per_cluster.plot.bar(stacked=True, figsize=(15, 8), cmap="tab20")

plt.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))

: 

# MICROGLIA DE

In [ ]:
# check if "log1p" is a key in .uns:
if "log1p" in adata_CD4.uns.keys():
    del adata_CD4.uns["log1p"]

sc.tl.rank_genes_groups(
    adata_CD4,
    groupby="leiden_nichevi",
    method="t-test",
    key_added="rank_genes",
    pts=True,
    use_raw=False,
)

sc.tl.rank_genes_groups(
    adata_CD4,
    groupby="leiden_nichevi",
    method="t-test",
    layer="corrupted_counts",
    key_added="rank_genes_corrupted",
    pts=True,
    use_raw=False,
)

CLUSTER = "8"

: 

In [ ]:
", ".join(list(adata.var_names))

: 

In [ ]:
# adata[:, 'Trem2'].X.sum()

: 

In [ ]:
sc.get.rank_genes_groups_df(adata_CD4, group=CLUSTER, key="rank_genes")

: 

In [ ]:
adata_CD4.uns["rank_genes"]["pts"]

: 

In [ ]:
sc.get.rank_genes_groups_df(adata_CD4, group="5", key="rank_genes_corrupted")

: 

In [ ]:
def compare_de_genes(
    adata: AnnData,
    key_uncorr: str,
    key_corr: str,
    group: str,
    log2fc_min: float | None = None,
    pval_cutoff: float | None = None,
    column_to_compare: str = "logfoldchanges",
    upregulated_in_corrupted: bool = False,
) -> pd.DataFrame:
    gene_df_1 = sc.get.rank_genes_groups_df(
        adata,
        key=key_uncorr,
        group=group,
        log2fc_min=None,
        pval_cutoff=None,
    )
    gene_df_2 = sc.get.rank_genes_groups_df(adata, key=key_corr, group=group, log2fc_min=None, pval_cutoff=None)

    df_compare = pd.merge(
        gene_df_1,
        gene_df_2,
        how="inner",
        on="names",
        suffixes=("_uncorr", "_corr"),
    )

    if upregulated_in_corrupted:
        df_subset = df_compare[df_compare[column_to_compare + "_uncorr"] < df_compare[column_to_compare + "_corr"]]

    else:
        df_subset = df_compare[df_compare[column_to_compare + "_uncorr"] > df_compare[column_to_compare + "_corr"]]

    df_subset = df_subset[df_subset["logfoldchanges_uncorr"] > log2fc_min]  # only positive logfoldchanges

    df_subset = df_subset[df_subset["pvals_adj_uncorr"] < 1e-3]  # only significant genes

    # remove the list of columns

    df_subset = df_subset[
        [
            "names",
            "logfoldchanges_uncorr",
            "logfoldchanges_corr",
            "pvals_adj_uncorr",
            "pvals_adj_corr",
            "pct_nz_group_corr",
            "pct_nz_reference_corr",
            "pct_nz_group_uncorr",
            "pct_nz_reference_uncorr",
        ]
    ]

    # sort by the logfoldchanges_uncorr

    df_subset = df_subset.sort_values(by="logfoldchanges_corr", ascending=False).reset_index(drop=True)

    return df_subset

: 

In [ ]:
compare_de_genes(
    adata_CD4,
    "rank_genes",
    "rank_genes_corrupted",
    group=CLUSTER,
    log2fc_min=0.5,
    pval_cutoff=0.5,
    upregulated_in_corrupted=True,
)

: 

In [ ]:
compare_de_genes(
    adata_CD4,
    "rank_genes",
    "rank_genes_corrupted",
    group=CLUSTER,
    log2fc_min=0.5,
    pval_cutoff=0.05,
    upregulated_in_corrupted=False,
).head(30)

: 

In [ ]:
gene_x = "Ptprc"
gene_y = "Gfap"
plt.scatter(
    adata_CD4[:, gene_x].layers["corrupted_counts"].toarray(),
    adata_CD4[:, gene_y].layers["corrupted_counts"].toarray(),
)
plt.scatter(
    adata_CD4[:, gene_x].X.toarray(),
    adata_CD4[:, gene_y].X.toarray(),
)

: 

In [ ]:
genes = compare_de_genes(
    adata_CD4,
    "rank_genes",
    "rank_genes_corrupted",
    group=CLUSTER,
    log2fc_min=0.5,
    pval_cutoff=0.05,
    upregulated_in_corrupted=False,
)["names"].to_list()

: 

In [ ]:
genes

: 

In [ ]:
sc.pl.dotplot(adata, genes, groupby=setup.CELL_TYPE, log=True, use_raw=False)

: 

In [ ]:
sc.pl.dotplot(adata_CD4, genes, groupby="leiden_nichevi", log=True, use_raw=False)

: 

In [ ]:
sc.pl.dotplot(
    adata_CD4,
    genes,
    groupby="leiden_nichevi",
    log=True,
    use_raw=False,
    layer="corrupted_counts",
)

: 

In [ ]:
genes_5 = sc.get.rank_genes_groups_df(
    adata_CD4,
    key="rank_genes",
    group="5",
    log2fc_min=1,
)

genes_5

: 

In [ ]:
genes_5_corrupted = sc.get.rank_genes_groups_df(
    adata_CD4,
    key="rank_genes_corrupted",
    group="5",
    log2fc_min=1,
)

genes_5_corrupted

: 

In [ ]:
from scvi.criticism import PosteriorPredictiveCheck

models_dict = {"nichevi": nichevae}

ppc = PosteriorPredictiveCheck(adata, models_dict, count_layer_key="counts", n_samples=10, indices=None)

: 